# Map Diseases Radboud -> MONDO

# Strategy

Diseases have been extracted from the three files that contain disease references.

This is disease.list.uniq

There's all kinds of namespaces in there... a bit of a mess!

Here we will try to map into MONDO.  In some cases, the disease reference is already MONDO, so that's good!  Other namespaces that I think we can mange are:  EFO (sometimes), DOID, and Orphanet





In [1]:
# puts `head -20 disease.list.uniq`
# puts `grep MONDO disease.list.uniq | head -20`
puts ENV["APIKEY"] 

74027bd8-6be0-4329-be22-aa3717f97243


In [79]:
require 'rest-client'
require 'json'

def map_id_to_disease(name)
    name.strip!
    warn "mapping #{name}"
  return false unless (name =~ /EFO_/ || name =~ /DOID_(.*)/ || name =~ /MONDO_(.*)/ || name =~ /Orphanet_(.*)/)
  identifier = $1  #W dfrom the match above
  api_key = ENV["APIKEY"] # Replace with BioPortal API key
  begin
    if name =~ /DOID/
      url =  "https://data.bioontology.org/search?q=DOID:#{identifier}&ontologies=DOID&require_exact_match=true&apikey=#{api_key}"
    elsif name =~ /MONDO/
      url =  "https://data.bioontology.org/search?q=MONDO:#{identifier}&ontologies=MONDO&require_exact_match=true&apikey=#{api_key}"      
    else
      url =  "https://data.bioontology.org/search?q=Orphanet:#{identifier}&ontologies=ORDO&require_exact_match=true&apikey=#{api_key}"
    end
#     warn url
    response = RestClient.get(url)
  rescue StandardError => e
    warn "No data found for #{name} #{e.inspect}\n"
    return false
  end
  mappings = []
  data = JSON.parse(response)

#  if hit = data.dig('collection', 0)
  if hit = data.dig('collection')
    hit.each do |h|
      disease_name = h&.dig('prefLabel')
      uri = h&.dig('@id')
      linksurl = h&.dig('links', 'mappings')
      mappings << { id: name, uri: uri, disease_name: disease_name, linksurl: linksurl}
    end
    return mappings
  else
    warn "No data found for #{name}\n"
    return false
  end

rescue StandardError => e
  warn "No data found for #{name} Error: #{e.inspect}\n"
  return false
end

# test
["DOID_7551", "MONDO_0003243", "Orphanet_10"].each do |name|
  warn map_id_to_disease(name)  # returns array
end

mapping DOID_7551
{:id=>"DOID_7551", :uri=>"http://purl.obolibrary.org/obo/DOID_7551", :disease_name=>"gonorrhea", :linksurl=>"https://data.bioontology.org/ontologies/DOID/classes/http%3A%2F%2Fpurl.obolibrary.org%2Fobo%2FDOID_7551/mappings"}
mapping MONDO_0003243
{:id=>"MONDO_0003243", :uri=>"http://purl.obolibrary.org/obo/MONDO_0003243", :disease_name=>"hepatocellular clear cell carcinoma", :linksurl=>"https://data.bioontology.org/ontologies/MONDO/classes/http%3A%2F%2Fpurl.obolibrary.org%2Fobo%2FMONDO_0003243/mappings"}
mapping Orphanet_10
{:id=>"Orphanet_10", :uri=>"http://www.orpha.net/ORDO/Orphanet_10", :disease_name=>"48,XXYY syndrome", :linksurl=>"https://data.bioontology.org/ontologies/ORDO/classes/http%3A%2F%2Fwww.orpha.net%2FORDO%2FOrphanet_10/mappings"}


["DOID_7551", "MONDO_0003243", "Orphanet_10"]

In [80]:
def get_mondo_from_mapping(mapping)
  api_key = ENV["APIKEY"] # Replace with BioPortal API key
  begin
    url = mapping+"?apikey=#{api_key}"
#     warn url
    response = RestClient.get(url)
  rescue StandardError => e
    warn "No data found for mapping #{e.inspect}\n"
    return false
  end
    mappings = []
    data = JSON.parse(response)

    data.each do |mapp|
      mapp["classes"].each do |thisclass|
        next unless thisclass["@id"] =~ /MONDO/
        return {"mondo": thisclass["@id"]}
      end
    end
    warn "no MONDO found for DOID/ORDO/MONDO #{mapping}"
    return false

rescue StandardError => e
  warn  "parsing or other error in #{mapping} Error: #{e.inspect}\n"
  return false
end

# test
["DOID_7551", "MONDO_0003243", "Orphanet_10"].each do |name|
  results = map_id_to_disease(name)  # this returns a hash
  if results
    results = results.first # (can only be one)
    nextres = get_mondo_from_mapping(results[:linksurl])
    if nextres
      results.merge! nextres
    end
    puts results
  else
    warn "found no results for #{name}"
  end
end

mapping DOID_7551


{:id=>"DOID_7551", :uri=>"http://purl.obolibrary.org/obo/DOID_7551", :disease_name=>"gonorrhea", :linksurl=>"https://data.bioontology.org/ontologies/DOID/classes/http%3A%2F%2Fpurl.obolibrary.org%2Fobo%2FDOID_7551/mappings", :mondo=>"http://purl.obolibrary.org/obo/MONDO_0004277"}


mapping MONDO_0003243


{:id=>"MONDO_0003243", :uri=>"http://purl.obolibrary.org/obo/MONDO_0003243", :disease_name=>"hepatocellular clear cell carcinoma", :linksurl=>"https://data.bioontology.org/ontologies/MONDO/classes/http%3A%2F%2Fpurl.obolibrary.org%2Fobo%2FMONDO_0003243/mappings", :mondo=>"http://purl.obolibrary.org/obo/MONDO_0003243"}


mapping Orphanet_10


{:id=>"Orphanet_10", :uri=>"http://www.orpha.net/ORDO/Orphanet_10", :disease_name=>"48,XXYY syndrome", :linksurl=>"https://data.bioontology.org/ontologies/ORDO/classes/http%3A%2F%2Fwww.orpha.net%2FORDO%2FOrphanet_10/mappings", :mondo=>"http://purl.obolibrary.org/obo/MONDO_0015028"}


["DOID_7551", "MONDO_0003243", "Orphanet_10"]

## Special Case EFO
sometimes can find efo -> MONDO

In [55]:
require 'rest-client'
require 'json'

def special_case_efo_to_mondo(efo)
        efo.strip!
    warn "**#{efo}**"

    api_key = ENV["APIKEY"]
    url = "https://data.bioontology.org/ontologies/EFO/classes/http%3A%2F%2Fwww.ebi.ac.uk%2Fefo%2F#{efo}?apikey=#{api_key}"
    warn url
    headers = {
    "Accept"     => "application/json",
    "User-Agent" => "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/137.0.0.0 Safari/537.36"
    }

    begin
        response = RestClient.get(url, headers: headers)
#         warn response
    rescue
        warn "step 1 EFO Lookup failed"
        return false
    end
    
    data = JSON.parse(response.body)
    mondo = ""

    if prefname = data.dig('prefLabel')
        warn "MATCHING #{prefname}"
        url2 = "https://data.bioontology.org/search?q=#{prefname}&ontologies=MONDO&apikey=#{api_key}"
#         warn "getting #{url2}"
        begin
            response = RestClient.get(url2, headers: headers)
        rescue
            warn "step 2 failed - search for MONDO by prefname HTTP call failed"
            return false
        end
        data = JSON.parse(response)
#         warn data
        records = data.dig('collection')
        records.each do |r|
#             warn r['prefLabel']
            next unless r['prefLabel'].downcase == prefname.downcase
            mondo = r['@id']
            break
        end
    else
        warn "Step 2 failed - can't find preflabel for #{efo}...???"
    end
    if mondo
        return {id: efo, mondo: mondo, disease_name: prefname}
    else
        warn "general failure - can't find mondo for #{efo} #{prefname} ...???"
        return false
    end
end
# test
puts special_case_efo_to_mondo("EFO_0000713")

**EFO_0000713**
https://data.bioontology.org/ontologies/EFO/classes/http%3A%2F%2Fwww.ebi.ac.uk%2Fefo%2FEFO_0000713?apikey=74027bd8-6be0-4329-be22-aa3717f97243
MATCHING subarachnoid hemorrhage


{:id=>"EFO_0000713", :mondo=>"http://purl.obolibrary.org/obo/MONDO_0005099", :disease_name=>"subarachnoid hemorrhage"}


## special case Orphanet
BioOntologies has a mapping from MONDO to Orphanet, but not vice versa

SO... if we have an Orphanet code, we need to get the prefLabel (disease name according to orphanet), then search MONDO by that disease name, and match it against the discovered MONDO recofrds.  UNFORTUNATELY, it often appears as a synonym, rather than the prefLabel for the record, so... that sucks!

Note - it is not the first or even the only match!  Need to parse through all matching MONDO records until we hit the exact match in the synonyms.


In [39]:
# require 'rest-client'
# require 'json'

# def special_case_orpha_to_mondo(orphacode)
#         orphacode.strip!

#     api_key = ENV["APIKEY"]
#     mapping = map_id_to_disease(orphacode).first  # returns array
# #     warn mapping.class
# # {:id=>"Orphanet_101010", :uri=>"http://www.orpha.net/ORDO/Orphanet_101010", :disease_name=>"Autosomal spastic paraplegia type 30", :linksurl=>"https://data.bioontology.org/ontologies/ORDO/classes/http%3A%2F%2Fwww.orpha.net%2FORDO%2FOrphanet_101010/mappings"}
#     search_term = mapping[:disease_name]
# #     warn search_term
#     unless search_term
#         warn "no disease name found for #{orphacode} (this is very strange!)"
#         return false
#     end
#     searchurl = "https://data.bioontology.org/search?q=#{search_term}&ontologies=MONDO&apikey=#{api_key}"
#     headers = {
#     "Accept"     => "application/json",
#     "User-Agent" => "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/137.0.0.0 Safari/537.36"
#     }

#     begin
#         response = RestClient.get(searchurl, headers: headers)
# #         warn response
#     rescue
#         return false
#     end
#     warn "matching #{search_term}"
#     mondo = ""
#     data = JSON.parse(response)
#     records = data.dig('collection')
#     records.each do |r|
# # prefLabel": "hereditary spastic paraplegia 30",
# # -"synonym": [
# # "KIF1A hereditary spastic paraplegia",
# # "hereditary spastic paraplegia type 30",
# # "autosomal spastic paraplegia type 30",
# # "hereditary spastic paraplegia caused by mutation in KIF1A",
# # "spastic paraplegia 30, autosomal dominant",
# # "SPG30"
# # ],
#         next unless r['synonym']
#         r['synonym'].each do |synonym|
# #             warn "match against #{synonym}"
#             next unless synonym.downcase == search_term.downcase
#             mondo = r['@id']
#         end
#     end
#     if mondo
#         return mapping.merge!({mondo: mondo})  #W preserve the existing hash
#     else
#         warn "general failure - can't find mondo for orphanet #{orphacode} #{search_term} ...???"

#         return false
#     end

# end

# # test
# ["Orphanet_101010"].each do |orphacode|
#   puts  special_case_orpha_to_mondo(orphacode)
# #     {:id=>"DOID_7551", :uri=>"http://purl.obolibrary.org/obo/DOID_7551", :disease_name=>"gonorrhea", :linksurl=>"https://data.bioontology.org/ontologies/DOID/classes/http%3A%2F%2Fpurl.obolibrary.org%2Fobo%2FDOID_7551/mappings", :mondo=>"http://purl.obolibrary.org/obo/MONDO_0004277"}
# end

In [89]:
require 'rest-client'
require 'json'

def special_case_orpha_to_mondo(orphacode)
    orphacode.strip!
#     warn "special case orphacode #{orphacode}"
    mapping = map_id_to_disease(orphacode).first  # returns array
#     warn mapping.class
    
# {:id=>"Orphanet_101010", :uri=>"http://www.orpha.net/ORDO/Orphanet_101010", :disease_name=>"Autosomal spastic paraplegia type 30", :linksurl=>"https://data.bioontology.org/ontologies/ORDO/classes/http%3A%2F%2Fwww.orpha.net%2FORDO%2FOrphanet_101010/mappings"}
    search_term = mapping[:disease_name]
#     warn search_term
    unless search_term
        warn "no disease name found for #{orphacode} (this is very strange!)"
        return false
    end
    
    orphacode.gsub!(/_/, ":")
    entity_ids = "entity_id=#{orphacode}&"

    response = RestClient.get(
    "https://api-v3.monarchinitiative.org/v3/api/mappings?#{entity_ids}format=json&limit=500&offset=0"
    )

#     warn "Request URL: #{response.request.url}"

    data = JSON.parse(response.body)
# warn data
    data['items'].each do |item|
        if item['object_id'] =~ /#{orphacode}/
          mondo_id = item['subject_id']  # e.g. "MONDO:0018940"
          mondo_term = item['subject_label']  # e.g. "Leigh Syndrome"
          return mapping.merge({mondo: mondo_id, term: mondo_term})
        end
    end
    warn "general failure - can't find mondo for orphanet #{orphacode} #{search_term} ...???"
    return false
end

# test
# ["Orphanet_101010"].each do |orphacode|
["Orphanet_10"].each do |orphacode|
  puts  special_case_orpha_to_mondo(orphacode)
#     {:id=>"DOID_7551", :uri=>"http://purl.obolibrary.org/obo/DOID_7551", :disease_name=>"gonorrhea", :linksurl=>"https://data.bioontology.org/ontologies/DOID/classes/http%3A%2F%2Fpurl.obolibrary.org%2Fobo%2FDOID_7551/mappings", :mondo=>"http://purl.obolibrary.org/obo/MONDO_0004277"}
end

mapping Orphanet_10


{:id=>"Orphanet:10", :uri=>"http://www.orpha.net/ORDO/Orphanet_10", :disease_name=>"48,XXYY syndrome", :linksurl=>"https://data.bioontology.org/ontologies/ORDO/classes/http%3A%2F%2Fwww.orpha.net%2FORDO%2FOrphanet_10/mappings", :mondo=>"MONDO:0015028", :term=>"48,XXYY syndrome"}


["Orphanet:10"]

In [83]:
require 'csv'

file = "disease.list.uniq"

diseaselist = File.readlines(file)

puts diseaselist.length  
puts diseaselist.first

9341
"DOID_10718"


In [74]:
puts `grep -i 'orphanet_' disease.list.uniq | wc -l`

0


In [28]:
puts `grep -i EFO disease.list.uniq | wc -l`

2829


In [ ]:
$stdout.sync = true   # Disables buffering for STDOUT
$stderr.sync = true   # Disables buffering for STDERR (warn, etc.)

# write the mapping file
f = File.open("./maps/diseases.map", "w")
e = File.open("./maps/diseases-errors.txt", "w")

f.write CSV.generate_line(["source","mondo","prefname"])

# spoeial case for EFO
diseaselist.each do |name|
    name.gsub!(/"/, "")
    next unless name =~ /EFO_/
    res = special_case_efo_to_mondo(name) # {:id=>"EFO_0000713", :mondo=>"http://purl.obolibrary.org/obo/MONDO_0005099", :disease_name=>"subarachnoid hemorrhage"}
    if !res || res[:mondo].empty?
        warn "found no mondo for #{name}"
        e.write "found no mondo for #{name}\n"
        e.flush  # This forces the write to disk immediately
        next
    end
    f.write CSV.generate_line([res[:id],res[:mondo],res[:disease_name]])
    f.flush
end


# spoeial case for ORPHA
diseaselist.each do |name|
    name.gsub!(/"/, "")
    next unless name =~ /Orphanet_/
    warn name
    
    res = special_case_orpha_to_mondo(name) # {:id=>"Orphanet:101010", :uri=>"http://www.orpha.net/ORDO/Orphanet_101010", :disease_name=>"Autosomal spastic paraplegia type 30", :linksurl=>"https://data.bioontology.org/ontologies/ORDO/classes/http%3A%2F%2Fwww.orpha.net%2FORDO%2FOrphanet_101010/mappings", :mondo=>"http://purl.obolibrary.org/obo/MONDO_0012476"}
    if !res || res[:mondo].empty?
        warn "found no mondo for #{name}"
        e.write "found no mondo for #{name}\n"
        e.flush
        next
    end
    warn "found mondo #{res[:mondo]}"
    f.write CSV.generate_line([res[:id],res[:mondo],res[:disease_name]])
    f.flush
end


diseaselist.each do |name|
    name.gsub!(/"/, "")
    next if name =~ /HP_/  # not HPO terms
    next if name =~ /EFO_/  # not EFO terms
    next if name =~ /OTAR_/  # not OTAR terms
    next if name =~ /ORPHA_/  # not ORPHA terms
  
  results = map_id_to_disease(name)
  if results
    results = results.first # (can only be one)
    nextres = get_mondo_from_mapping(results[:linksurl])
    if nextres
      results.merge! nextres
    else
      warn "found no mondo for #{name}"
      e.write "found no mondo for #{name}\n"
      e.flush
    end

    f.write CSV.generate_line([results[:id],results[:mondo],results[:disease_name]])
    f.flush
    # {:id=>"Orphanet_101010", :uri=>"http://www.orpha.net/ORDO/Orphanet_101010", :disease_name=>"Autosomal spastic paraplegia type 30", :linksurl=>"https://data.bioontology.org/ontologies/ORDO/classes/http%3A%2F%2Fwww.orpha.net%2FORDO%2FOrphanet_101010/mappings", :snomed=>"http://purl.bioontology.org/ontology/SNOMEDCT/763377006"}
  else
    warn "found no results for #{name}"
    e.write "found no results for #{name}\n"
    e.flush
  end
end
f.close
e.close

**EFO_0000174**
https://data.bioontology.org/ontologies/EFO/classes/http%3A%2F%2Fwww.ebi.ac.uk%2Fefo%2FEFO_0000174?apikey=74027bd8-6be0-4329-be22-aa3717f97243
MATCHING Ewing sarcoma
**EFO_0000180**
https://data.bioontology.org/ontologies/EFO/classes/http%3A%2F%2Fwww.ebi.ac.uk%2Fefo%2FEFO_0000180?apikey=74027bd8-6be0-4329-be22-aa3717f97243
MATCHING HIV-1 infection
found no mondo for EFO_0000180
**EFO_0000195**
https://data.bioontology.org/ontologies/EFO/classes/http%3A%2F%2Fwww.ebi.ac.uk%2Fefo%2FEFO_0000195?apikey=74027bd8-6be0-4329-be22-aa3717f97243
MATCHING metabolic syndrome
found no mondo for EFO_0000195
**EFO_0000196**
https://data.bioontology.org/ontologies/EFO/classes/http%3A%2F%2Fwww.ebi.ac.uk%2Fefo%2FEFO_0000196?apikey=74027bd8-6be0-4329-be22-aa3717f97243
MATCHING metastatic prostate cancer
found no mondo for EFO_0000196
**EFO_0000205**
https://data.bioontology.org/ontologies/EFO/classes/http%3A%2F%2Fwww.ebi.ac.uk%2Fefo%2FEFO_0000205?apikey=74027bd8-6be0-4329-be22-aa3717f97243


In [1]:
puts `pwd`



/home/osboxes/CODE/SIMPATHIC2/SKG_Mapping/radboud
